Description: Get peak information, perform coincidence, and plot summed spectrum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import glob
import os
sys.path.insert(0,"/home/ws/sk6801/sw/UCSD_analysis/sandpro")
import sandpro
import configparser
import json
import scipy.stats
from matplotlib.colors import LogNorm

from scipy.optimize import curve_fit
import datetime
import pandas as pd
from copy import deepcopy
from numba import jit
import time

sys.path.insert(0,"../src/")
import common.d2d as d2d
import common.utils as util
import data_processing.fast_processor_all_channel as fast_processor
from data_processing.event_processor_all_channel import EventProcessor

from data_structure.waveform_info import WaveformInfo
from data_structure.peak_info import PeakInfo


In [ ]:
class PeakInfoTest():
    def __init__(self, start_time_array, end_time_array, peak_max_array, area_array):
        self.start_time_array = start_time_array
        self.end_time_array = end_time_array
        self.peak_max_array = peak_max_array
        self.area_array = area_array
        self.peak_count = len(start_time_array)
        


In [ ]:
def get_baseline_for_all_events(waveform, baseline_front=(0.0,0.2)):
    baseline_start_f = int(1000 * baseline_front[0])
    baseline_end_f = int(1000 * baseline_front[1])

    # baseline is calculated with raw waveform
    # unit: same as raw waveform
    baseline_mean_V = np.mean(waveform[:,baseline_start_f:baseline_end_f], axis=1)
    baseline_std_V = np.std(waveform[:,baseline_start_f:baseline_end_f], axis=1)

    return baseline_mean_V, baseline_std_V
        

In [ ]:
def rolling_window(array: np.ndarray, window_size:int, axis:int) -> np.ndarray:
    """
    Rolls a 1D array into a 2D array with a sliding window view.
    Args:
        array (np.ndarray): The input 1D array to be rolled.
        window_size (int): The size of the rolling window.
        axis (int): The axis along which to roll the array.
    Returns:
        np.ndarray: A 2D array where each row corresponds to a window of the original array.
    """
    ndim = array.ndim

    if not isinstance(array, np.ndarray):
        raise ValueError("Input must be a numpy array.")
    if axis > ndim - 1 or axis < 0:
        raise ValueError("Axis must be within the range of the array dimensions.")
    if not isinstance(window_size, int) or window_size <= 0 or window_size > array.shape[axis]:
        raise ValueError("Window size must be a positive integer.")
    
    # n.dim rolling window
    # expand array according to the rolling window
    expanded_array = np.lib.stride_tricks.sliding_window_view(array, window_size, axis=axis)
    # take the mean along the new dimension for the result
    roll_averaged_array = expanded_array.mean(axis=ndim) 

    # roll_averaged_array = np.convolve(array, np.ones(window_size)/window_size, mode='valid')


    return roll_averaged_array

In [ ]:
# def get_peaks(
#         waveform: np.array, 
#         event_time_s, 
#         window_size = 4, 
#         threshold_sig = 5, 
#         peak_width_sample = 30):
#     # waveform is a 2D array, with shape (n_events, m_samples)
#     # event_time_s is the time of the event in seconds, not used in this function
#     # window_size is the size of the rolling window to smooth the data
#     # threshold_sig is the number of standard deviations above the baseline to consider a peak
#     # peak_width_sample is the minimum width of the peak in samples

#     # check if waveform is a 2D array
#     if waveform.ndim != 2:
#         raise ValueError("Waveform must be a 2D array with shape (n_events, m_samples)")
    
#     # creating smooth waveform by applying a rolling window
#     # rolling window in sample axis, which is axis 1
#     smooth_waveform = rolling_window(waveform, window_size, axis=1)
    
#     # rmb to change waveform to filtered waveform
#     # add rolling window to smooth the data, roll every 3 points
#     # test_averaged = np.convolve(waveform, np.ones(window_size)/window_size, mode='valid')

#     # recalculated the baseline for the smoothed waveform
#     baseline, baseline_std = get_baseline_for_all_events(smooth_waveform)
#     threshold = baseline + threshold_sig * baseline_std

#     # applying the threshold to find peaks
#     mask = smooth_waveform > threshold

#     # mark the start and end of the peaks
#     diff = np.diff(mask, axis = 1)
#     points = np.where(diff == 1)[0]
#     # start points are the points after the rising edge
#     points[0::2] = points[0::2]+1 

#     # # define the minimum peak width in samples
#     # min_peak_width_sample = window_size*2

#     # pair the start and end points of the peaks
#     # remove the last point if it's odd
#     if len(points) % 2 != 0:
#         points = points[:-1]  
#     # reshape the points into pairs
#     pairs = points.reshape(-1, 2)

#     # remove the pair if they are too close to each other
#     for pair in pairs:
#         if pair[1] - pair[0] < peak_width_sample:
#             pairs = np.delete(pairs, np.where((pairs == pair).all(axis=1)), axis=0)

#     # flatten the pairs be better handling
#     points = pairs.ravel()

#     # results
#     pairs_ns = pairs * 4 # in ns, assuming the sampling rate is 250 MHz (4 ns per sample)
#     start_time_array, end_time_array = pairs_ns[:,0], pairs_ns[:,1]

#     peak_max_array = np.empty(pairs.shape[0], dtype=float)
#     area_array = np.empty(pairs.shape[0], dtype=float)

#     for i, pair in enumerate(pairs):
#         peak_max_array[i] = np.max(waveform[pair[0]:pair[1]])
#         area_array[i] = np.sum(waveform[points[0]:points[1]])

#     return start_time_array, end_time_array, peak_max_array, area_array

# v_get_waveform = np.vectorize(
#     get_peaks, 
#     excluded=['window_size', 'threshold_sig', 'peak_width_sample'], 
#     signature="(n,m) -> (n)")

In [ ]:
def get_board_channel(SiPM_channel: int, board_0_channels: np.array, board_1_channels: np.array) -> int:
    if SiPM_channel in board_0_channels: 
        board_channel = np.where(board_0_channels == SiPM_channel)[0]
    elif SiPM_channel in board_1_channels:  
        board_channel = np.where(board_1_channels == SiPM_channel)[0]
    else:
        raise ValueError(f"SiPM channel {SiPM_channel} not found in both boards.")

    return board_channel[0]


In [ ]:
def set_peak_info_from_waveform_info(waveform_info: WaveformInfo):
    PeakInfoList = []
    peak_info = PeakInfo()
    peak_info.set_info_from_dict(waveform_info.__dict__)

    for peak_id in range(int(waveform_info.n_peaks)):
        peak_info.peak_id = peak_id
        peak_info.peak_start_time_s = waveform_info.peak_start_time_s_array[peak_id]
        peak_info.peak_end_time_s = waveform_info.peak_end_time_s_array[peak_id]
        peak_info.peak_rel_start_time_s = waveform_info.peak_rel_start_time_s_array[peak_id]
        peak_info.peak_height_V = waveform_info.peak_height_V_array[peak_id]
        peak_info.peak_width_ns = waveform_info.peak_width_ns_array[peak_id]
        peak_info.peak_area_Vns = waveform_info.peak_area_Vns_array[peak_id]
        peak_info.peak_area_PE = waveform_info.peak_area_PE_array[peak_id]

        tmp  = deepcopy(peak_info.__dict__)
        PeakInfoList.append(tmp)

    return PeakInfoList

In [ ]:
board_0_channels = np.array([0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15])
board_1_channels = np.array([16,17,18,19,20,21,22,23])

get_board_channel(16, board_0_channels, board_1_channels)

### Example

In [ ]:
df = pd.read_csv("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250616_LXe_gain_info_single_channel.csv", 
                 parse_dates=["date_time"],
                 delimiter=",",
                 quotechar='"', 
                 skipinitialspace=True, 
                 encoding="utf-8")
df['board_0_channels'] = df['board_0_channels'].apply(json.loads).apply(np.array)
df['board_1_channels'] = df['board_1_channels'].apply(json.loads).apply(np.array)


In [ ]:
all_runs_d2d = d2d.data(df)
mask_data_taking_mode = (all_runs_d2d.data_taking_mode == "all_channels")
mask_gain_nan = ~np.isnan(all_runs_d2d.gain)
mask = mask_data_taking_mode & mask_gain_nan

all_runs_d2d.apply_mask(mask, inplace=True, dry = False)
all_run_list = np.unique(all_runs_d2d.md_full_path)

In [ ]:
for channel in range(24):
    mask = all_runs_d2d.channel == channel
    test = all_runs_d2d.apply_mask(mask, inplace=False, dry = False)

    plt.plot(test.date_time, test.spe_position, '-o', label=f'Channel {channel}')

# rotate x-axis labels
plt.xticks(rotation=45)

In [ ]:
count_successful = 0
count_failed = 0

for i, md_full_path in enumerate(all_run_list):

    mask = all_runs_d2d.md_full_path == md_full_path
    single_run = all_runs_d2d.apply_mask(mask, inplace=False, dry = True)

    if len(single_run.channel) < 24:
        # print(f"Run {md_full_path} has {len(single_run.channel)} channels, expected 24 channels.")
        count_failed += 1

        continue    
    elif len(single_run.channel) == 24:
        print(f"Run {i}: {md_full_path} has 24 channels. Run tag: {single_run.run_tag[0]}")
        count_successful += 1
        
    else: 
        raise ValueError(f"Run {md_full_path} has {len(single_run.channel)} channels, expected 24 channels.")

print(f"Total runs: {len(all_run_list)}"
      f"Successful runs: {count_successful} "
      f"Failed runs: {count_failed} "
      f"Success rate: {count_successful/len(all_run_list)*100:.2f}%")

In [ ]:
def get_peak_level_data(all_runs_d2d: d2d.data,
                        md_full_path: str):
    
    single_info = WaveformInfo()
    single_info_list = []
    
    mask = all_runs_d2d.md_full_path == md_full_path
    # print(f"Processing run {md_full_path}")
    single_run = all_runs_d2d.apply_mask(mask, inplace=False, dry = False)
    # print(single_run.__dict__)

    assert len(single_run.channel) == 24, f"Run {md_full_path} has {len(single_run.channel)} channels, expected 24 channels."
    
    t1,t2,t3, t4 = 0, 0, 0, 0

    for board_id in np.unique(single_run.board):
        
        mask = single_run.board == board_id
        single_board = single_run.apply_mask(mask, inplace=False, dry = True)

        board_info = WaveformInfo()
        board_info.set_info_from_dict(single_board.get_common_info_dict())
        board_info.board_0_channels = np.array(json.loads(board_info.board_0_channels))
        board_info.board_1_channels = np.array(json.loads(board_info.board_1_channels))

        # single_info.set_info_from_dict(board_info)
        event_processor = EventProcessor(board_info)
        
        if board_id == 0:
            channel_list = board_info.board_0_channels
        else:
            channel_list = board_info.board_1_channels


        for channel_id in range(len(channel_list)):
            mask = (single_run.board == board_id) & (single_run.channel == channel_list[channel_id])
            single_channel = single_run.apply_mask(mask, inplace=False, dry = True)
            
            assert len(single_channel.channel) == 1, f"Channel {channel_id} in board {board_id} has {len(single_channel.channel)} channels, expected 1 channel."
            
            single_info.set_info_from_dict(single_channel.get_dict())
            board_channel = get_board_channel(channel_id, board_info.board_0_channels, board_info.board_1_channels)

            waveform_processor = event_processor.get_waveform_processor(board_channel)
            waveform = waveform_processor.filtered_wfs

            baseline, baseline_std = get_baseline_for_all_events(waveform)

            # print(f"Processing board {single_info.board}, channel {single_info.channel} in run {single_info.md_full_path}")

            for event_id in range(waveform.shape[0]):
                single_waveform = waveform[event_id,:]
                single_baseline, single_baseline_std = baseline[event_id], baseline_std[event_id]

                single_info.event_start_time_s = waveform_processor.event_time_s[event_id]
                single_info.event_id = event_id
                # print(f"Processing event {single_info.event_id} in run {single_info.md_full_path}, board {single_info.board}, channel {single_info.channel}")

                t0 = time.perf_counter()
                single_info.set_peaks_for_single_processed_waveform(
                    single_waveform, 
                    single_baseline, 
                    single_baseline_std,
                    threshold_sig=5, 
                    extend_sum_window=50,
                    # event_id=event_id
                )
                t1 += time.perf_counter() - t0              

                t0 = time.perf_counter()
                PeakInfoList = set_peak_info_from_waveform_info(single_info)
                t2 += time.perf_counter() - t0              

                # single_info_list.append(single_info.__dict__.copy())
                # tmp = deepcopy(single_info.__dict__)
                # single_info_list.append(tmp)

                t0 = time.perf_counter()
                single_info_list += PeakInfoList
                t3 += time.perf_counter() - t0

    print(f'Time taken: {t1:.6f} seconds for peak finding, '
          f'{t2:.6f} seconds for peak info setting, '
          f'{t3:.6f} seconds for appending peak info list')
    
    return (single_info_list, waveform, baseline, baseline_std)

In [ ]:
def get_waveform_from_single_info(single_info):
    
    # assert len(single_info.channel) == 1, f"Expected 1 channel."
    event_processor = EventProcessor(single_info)
            
    board_channel = get_board_channel(single_info.channel, single_info.board_0_channels, single_info.board_1_channels)

    waveform_processor = event_processor.get_waveform_processor(board_channel)
    waveform = waveform_processor.filtered_wfs
    baseline, baseline_std = get_baseline_for_all_events(waveform)

    return (waveform, baseline, baseline_std)

In [ ]:
df_result = pd.DataFrame(columns=WaveformInfo().__dict__.keys())
single_info_list = []

for md_full_path in all_run_list[79:80]:
    result, waveform, baseline, baseline_std = get_peak_level_data(all_runs_d2d=all_runs_d2d,
                                           md_full_path=md_full_path)
    single_info_list += result

df_result = pd.DataFrame.from_dict(single_info_list)
d2d_data = d2d.data(df_result)


In [ ]:
plt.close()
fig_peak_integral_area, axes_peak_integral_area = plt.subplots(3,8,figsize=(35,15))
fig_area_height, axes_area_height = plt.subplots(3,8,figsize=(35,15))
fig_area_width, axes_area_width = plt.subplots(3,8,figsize=(35,15))
fig_area_height_full, axes_area_height_full = plt.subplots(3,8,figsize=(35,15))
fig_area_width_full, axes_area_width_full = plt.subplots(3,8,figsize=(35,15))
fig_width_height, axes_width_height = plt.subplots(3,8,figsize=(35,15))

axes = [axes_area_height, axes_area_height_full, axes_area_width, axes_area_width_full, axes_width_height, axes_peak_integral_area]
figures = [fig_area_height, fig_area_height_full, fig_area_width, fig_area_width_full, fig_width_height, fig_peak_integral_area] 
axes_name = ['axes_area_height', 
            'axes_area_height_full', 
            'axes_area_width', 
            'axes_area_width_full', 
            'axes_width_height', 
            'axes_peak_integral_area']

# data selection
# mask = (d2d_data.peak_area_PE > 1.5) 
# mask = (d2d_data.peak_height_V < 1.2) 
# mask = (d2d_data.peak_width_ns < 300) & (d2d_data.peak_width_ns < 750) & (d2d_data.peak_width_ns > 500)
# mask = (d2d_data.peak_width_ns > 300)
# selected_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
selected_data = d2d_data

for channel in range(24):
    mask = selected_data.channel == channel
    singl_channel_data = selected_data.apply_mask(mask, inplace=False, dry=True)

    # # this also remove null values from the peak_height_V_array
    # array = list(singl_channel_data.peak_height_V_array)
    # height_V = np.concatenate(array)
    # array = list(singl_channel_data.peak_area_PE_array)
    # area_Vns = np.concatenate(array)
    height_V = singl_channel_data.peak_height_V
    area_PE = singl_channel_data.peak_area_PE
    width_ns = singl_channel_data.peak_width_ns
    integral_window_area_PE = list(singl_channel_data.integral_window_area_PE)
    integral_window_area_PE = np.concatenate(integral_window_area_PE)

    plot_row = channel % 3
    plot_col = channel // 3

    # change the rows so that it match with the physical layout of the channels
    if plot_row == 0:
        plot_row = 1
    elif plot_row == 1:
        plot_row = 0

    axes_peak_integral_area[plot_row,plot_col].hist((area_PE),
                            bins=100,
                            range=[-0.1,10], 
                            alpha=0.5,
                            label='peak_area_PE_array')
    axes_peak_integral_area[plot_row,plot_col].hist((integral_window_area_PE),
                            bins=100,
                            range=[-0.1,10], 
                            alpha=0.5,
                            label='integral_window_area_PE')   
    axes_peak_integral_area[plot_row,plot_col].set_yscale('log')
    if plot_row == 2:
        axes_peak_integral_area[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_peak_integral_area[plot_row,plot_col].set(ylabel='Counts')

    axes_area_height[plot_row,plot_col].hist2d(area_PE,height_V,
                            bins=[100,100],
                            range=[[-0.1,10],[0,0.1]],
                            cmap='viridis',
                            norm=LogNorm())
    if plot_row == 2:
        axes_area_height[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_area_height[plot_row,plot_col].set(ylabel='Height [V]')


    axes_area_width[plot_row,plot_col].hist2d(area_PE,width_ns,
                            bins=[100,100],
                            range=[[-0.1,10],[0,1000]],
                            cmap='viridis',
                            norm=LogNorm())  
    if plot_row == 2:
        axes_area_width[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_area_width[plot_row,plot_col].set(ylabel='Width [ns]')

    
    axes_area_height_full[plot_row,plot_col].hist2d(area_PE,height_V,
                            bins=[100,100],
                            range=[[-0.1,1000],[0,1.5]],
                            cmap='viridis',
                            norm=LogNorm())
    if plot_row == 2:
        axes_area_height_full[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_area_height_full[plot_row,plot_col].set(ylabel='Height [V]')


    axes_area_width_full[plot_row,plot_col].hist2d(area_PE,width_ns,
                            bins=[50,50],
                            range=[[-0.1,1000],[0,2500]],
                            cmap='viridis',
                            norm=LogNorm())  
    if plot_row == 2:
        axes_area_width_full[plot_row,plot_col].set(xlabel='Area [PE]')
    if plot_col == 0:
        axes_area_width_full[plot_row,plot_col].set(ylabel='Width [ns]')
    
    
    axes_width_height[plot_row,plot_col].hist2d(width_ns,height_V,
                            bins=[50,50],
                            range=[[0,1000],[0,1.5]],
                            cmap='viridis',
                            norm=LogNorm())  
    if plot_row == 2:
        axes_width_height[plot_row,plot_col].set(xlabel='Width [ns]')
    if plot_col == 0:
        axes_width_height[plot_row,plot_col].set(ylabel='Height [V]')
    
    for ax in axes:
        ax[plot_row,plot_col].set_title(f"Channel {channel}")

# Set plot title
for fig, ax_name in zip(figures, axes_name):
    fig.suptitle(ax_name)

# Move title upwards
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.legend(axes_peak_integral_area, loc='upper right', fontsize='small')
axes_peak_integral_area[2,7].legend(loc="upper right")


plt.show()

In [ ]:
plt.close()
fig, axes = plt.subplots(3,8,figsize=(35,15))

# masking
for channel in range(24):
    mask = d2d_data.channel == channel
    singl_channel_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

    # # this also remove null values from the peak_height_V_array
    # array = list(singl_channel_data.peak_height_V_array)
    # height_V = np.concatenate(array)
    # array = list(singl_channel_data.peak_area_PE_array)
    # area_Vns = np.concatenate(array)
    height_V = singl_channel_data.peak_height_V

    plot_row = channel % 3
    plot_col = channel // 3

    # change the rows so that it match with the physical layout of the channels
    if plot_row == 0:
        plot_row = 1
    elif plot_row == 1:
        plot_row = 0

    axes[plot_row,plot_col].hist(height_V,
                            bins=100,
                            range=[0,0.1])
    
    axes[plot_row,plot_col].set_title(f"Channel {channel}")

# set the labels
for ax in axes.flat:
    ax.set(xlabel='Height [V]', ylabel='Count')
    # log y
    ax.set_yscale('log')
    # ax.set_xlim(-0.1, 100)

# Set plot title
# plt.suptitle(f"Dataset: {path.split('/')[-1]}")

# Move title upwards
plt.tight_layout(rect=[0, 0.03, 1, 0.95])


plt.show()

### Test waveform

In [ ]:
mask = (d2d_data.peak_width_ns < 300) & (d2d_data.peak_height_V < 1.2)  & (d2d_data.peak_height_V > 0.2) & (d2d_data.channel == 14)
# mask = (d2d_data.peak_area_PE > 1e100) 
# mask = (d2d_data.peak_width_ns > 300) & (d2d_data.peak_height_V > 0.4) & (d2d_data.channel == 14)
# mask = (d2d_data.peak_width_ns > 400) & (d2d_data.peak_width_ns < 750) & (d2d_data.peak_width_ns > 500) & (d2d_data.channel == 11)
# mask = (d2d_data.peak_height_V > 1.2) & (d2d_data.channel == 11)
selected_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

single_data = selected_data.get_row_info_to_dict(0)
single_info = WaveformInfo()
single_info.set_info_from_dict(single_data)
waveform, baseline, baseline_std = get_waveform_from_single_info(single_info)

event_id_array = selected_data.event_id


In [ ]:
selected_data.peak_area_PE[(np.where(selected_data.event_id == 5510))]

In [ ]:
event_id_id = np.random.randint(0, len(event_id_array), size=1)[0]
event_id = event_id_array[event_id_id]

# event_id = 5510 

print(event_id)

single_processed_waveform = waveform[event_id,:]
single_baseline = baseline[event_id]
single_baseline_std = baseline_std[event_id]
threshold_sig=5
extend_sum_window=50
        
window_size = 6
min_peak_width_sample = window_size*3

smooth_waveform = np.convolve(
    single_processed_waveform, 
    np.ones(window_size)/window_size, mode='valid')

# recalculated the baseline for the smoothed waveform
threshold = single_baseline + threshold_sig * single_baseline_std
# threshold = 0.015

# applying the threshold to find peaks
mask = smooth_waveform > threshold

# mark the start and end of the peaks
diff = np.diff(mask)

samples_above_threshold = np.where(diff == 1)[0]
# start samples_above_threshold are the samples_above_threshold after the rising edge
# samples_above_threshold[0::2] = samples_above_threshold[0::2]+1 
samples_above_threshold[1::2] = samples_above_threshold[1::2] + window_size

# remove the last point if it's odd
if len(samples_above_threshold) % 2 != 0:
    samples_above_threshold = samples_above_threshold[:-1]  
    
# reshape the points into pairs for better handling
peak_boundaries = samples_above_threshold.reshape(-1, 2)

# remove the pair if they are too close to each other
peak_width = peak_boundaries[:,1] - peak_boundaries[:,0]
tmp = np.where(peak_width < min_peak_width_sample)
peak_boundaries = np.delete(peak_boundaries, tmp, axis=0)

# remove overlapping peaks
peak_boundaries[1:,0] < peak_boundaries[:-1,1]



fig, ax = plt.subplots(figsize=(10,6))
print(f"Threshold: {threshold:.3f} V")
# ax.plot(mask*max(smooth_waveform), 'g--', label='mask')
ax.plot(single_processed_waveform, label='raw waveform')
ax.plot(smooth_waveform, label='smoothed waveform')
ax.scatter(peak_boundaries, single_processed_waveform[peak_boundaries],  s = 50, marker = '*', color = 'black', label='smoothed waveform', zorder = 10)
ax.axhline(single_baseline, color='green', linestyle = 'dashed', label='baseline')
ax.axhline(threshold, color='black', linestyle = 'dashed', label='threshold')
print(f"Found {len(peak_boundaries)} peaks in event {event_id}.")

# avoid overlapping peaks
end_sample_of_previous_peak = 0

for peak_id, peak_boundary in enumerate(peak_boundaries):

    # find the first sample below the baseline before the peak boundary
    start_sample = peak_boundary[0] - np.where(single_processed_waveform[peak_boundary[0]::-1] - single_baseline < 0)[0][0]
    # find the first sample above the baseline after the peak boundary
    end_sample = peak_boundary[1] + np.where(single_processed_waveform[peak_boundary[1]:] - single_baseline < 0)[0][0]
    
    # skip if the start sample is before the end sample of the previous peak
    if start_sample < end_sample_of_previous_peak:
        continue

    end_sample_of_previous_peak = end_sample    
    
    peak_area_Vsample = np.sum(single_processed_waveform[start_sample:end_sample])
    peak_area_Vns = peak_area_Vsample * 4  # convert to V*ns, assuming 250 MHz sampling rate (4 ns per sample)  
    peak_area_PE = peak_area_Vns / single_info.spe_position
    peak_width_ns = (end_sample - start_sample)*4
    peak_height_V = np.max(single_processed_waveform[start_sample:end_sample])

    print(f"Peak {peak_id}:"
          f"peak_height_V = {peak_height_V:.3f}, peak_area_PE = {peak_area_PE:.3f}, "
          f"peak_width_ns = {peak_width_ns:.3f} \n")

    ax.plot([start_sample,end_sample],
        [single_processed_waveform[start_sample],single_processed_waveform[end_sample]], 
        'o', label = f"Peak {peak_id}, area[PE] = {peak_area_PE}")
    # ax.plot([peak_boundary[1],peak_boundary[0]],
    #     [single_processed_waveform[peak_boundary[1]],single_processed_waveform[peak_boundary[0]]], 
    #     'o', label = f"Peak boundary")


    

plt.title(f"Event {event_id}")
plt.legend()
plt.xlabel('Sample')
plt.ylabel('Amplitude [V]')
plt.show()


    # continue_bool = input("Press Enter to continue...")  # wait for user input to continue to the next event
    # if continue_bool.lower() == 'n':
    #     break
    # else:
    #     continue


In [ ]:
mask = selected_data.event_id == event_id
test = selected_data.apply_mask(mask, inplace=False, dry=True)

print(test.peak_area_PE)
# print(f"peak_height_V = {test.peak_height_V}, peak_area_PE = {test.peak_area_PE}, "
#         f"peak_width_ns = {test.peak_width_ns} \n")


In [ ]:
result, edge = np.histogram(d2d_data.peak_height_V, bins = 100, range=[0,0.1])
edge[np.argmax(result)]  # get the bin center of the peak

### Event Rate

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
# this also remove null values from the peak_height_V_array
array = list(d2d_data.peak_start_time_s_array)
peak_start_time = np.concatenate(array)

event_max = np.max(peak_start_time)
event_min = np.min(peak_start_time)
n_bins = int(event_max - event_min) # 1 second per bin

ax.hist(peak_start_time, range = [event_min,event_min+n_bins], bins = n_bins, alpha=0.5)

ax.set_xlabel('Time [s]')
ax.set_ylabel('Event Rate [Hz]')

plt.show()

In [ ]:
event_rate,bin_edge = np.histogram(peak_start_time, range = [event_min,event_min+n_bins], bins = n_bins)

In [ ]:
DetectorHeight = 100 #mm
DetectorDiameter = 100 #mm
DetectorLongestLength = (DetectorHeight**2 + DetectorDiameter**2)**0.5

SpeedofLight = 299792458 # m/s
CoincidenceTime = DetectorLongestLength / SpeedofLight * 1e9 # in ns

In [ ]:
CoincidenceTime

### Time difference between board 0 and 1

In [ ]:
mask = d2d_data.board == 0
single_board_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

board_0_peak_time = single_board_data.peak_rel_start_time_s

mask = d2d_data.board == 1
single_board_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
board_1_peak_time = single_board_data.peak_rel_start_time_s

length = np.min([len(board_0_peak_time), len(board_1_peak_time)])
time_diff = np.abs(board_1_peak_time[:length] - board_0_peak_time[:length]) # in s

# plt.hist(time_diff, bins=50, range = [0, 1e-7])
plt.hist(time_diff, bins=50, range=[1.5e-7,0.5e-6])
plt.show()


In [ ]:
count, edge = np.histogram(time_diff, bins=50, range=[1.5e-7,0.5e-6])
edge[np.argmax(count)]  # get the bin center of the peak

In [ ]:
# @jit(nopython=True)
def get_sum_area_PE_in_time_window(
    peak_start_time_s: np.ndarray, 
    relative_start_time: np.ndarray, 
    event_time: np.ndarray, 
    channel: np.ndarray, 
    board: np.ndarray, 
    peak_area_PE: np.ndarray,
    event_id: int,
    time_window_width: float,
    coincidence: int,
    ):
    """
    Calculate the sum of peak area for coincidental signals within a specified time window.
    
    Parameters:
    """
    counts = 0
    max_bin_edge = np.max(peak_start_time_s)
    min_bin_edge = np.min(peak_start_time_s)
    n_bins = int((max_bin_edge - min_bin_edge)/time_window_width) # 1 second per bin
    event_in_window,bin_edge = np.histogram(peak_start_time_s, range = [min_bin_edge,max_bin_edge], bins = n_bins)
    
    sum_area_PE_list = []
    rel_time = []
    mask = event_in_window > coincidence
    start_time_window = bin_edge[:-1][mask]
    end_time_window = start_time_window + time_window_width

    for start_time, end_time in zip(start_time_window, end_time_window):
        mask = peak_start_time_s >= start_time
        mask &= peak_start_time_s < end_time
        channels_within_window = channel[mask]
        board_within_window = board[mask]
        area_PE_within_window = peak_area_PE[mask]
        event_id_within_window = event_id[mask]
        event_time_s_within_window = event_time[mask]
        rel_time_within_window = relative_start_time[mask]
        
        if (len(np.unique(board_within_window)) == 2) & (len(np.unique(channels_within_window)) >= coincidence):
            # if (len(np.unique(channels_within_window)) >= coincidence):
            sum_area_PE = np.sum(area_PE_within_window)
            sum_area_PE_list.append(sum_area_PE)
            rel_time.append(np.min(event_time_s_within_window))
            
    return (sum_area_PE_list, rel_time)



In [ ]:
# @jit(nopython=True)
def get_counts_in_time_window(
    peak_start_time_s: np.ndarray, 
    relative_start_time: np.ndarray, 
    event_time: np.ndarray, 
    channel: np.ndarray, 
    board: np.ndarray, 
    peak_area_PE: np.ndarray,
    event_id: int,
    time_window_width: float,
    coincidence: int,
    ):
    """
    Calculate the sum of peak area for coincidental signals within a specified time window.
    
    Parameters:
    """
    counts = 0
    max_bin_edge = np.max(peak_start_time_s)
    min_bin_edge = np.min(peak_start_time_s)
    n_bins = int((max_bin_edge - min_bin_edge)/time_window_width) # 1 second per bin
    event_in_window,bin_edge = np.histogram(peak_start_time_s, range = [min_bin_edge,max_bin_edge], bins = n_bins)
    
    mask = event_in_window > coincidence
    start_time_window = bin_edge[:-1][mask]
    end_time_window = start_time_window + time_window_width

    for start_time, end_time in zip(start_time_window, end_time_window):
        mask = peak_start_time_s >= start_time
        mask &= peak_start_time_s < end_time
        channels_within_window = channel[mask]
        board_within_window = board[mask]
        
        if (len(np.unique(board_within_window)) == 2) & (len(np.unique(channels_within_window)) >= coincidence):
            counts += 1
            
    return counts



In [ ]:
# change peak_start_time_s for all board 1 in d2d_data
count_list = []
delay_time_list = np.arange(-10,10,1)

for delay_time in delay_time_list:
    change_time = d2d_data.get_df()
    change_time.loc[change_time.board == 1, 'peak_start_time_s'] += delay_time
    d2d_data_updated = d2d.data(change_time)
    mask = (d2d_data_updated.peak_area_PE > d2d_data_updated.spe_position*1.5)
    clean_data = d2d_data_updated.apply_mask(mask, inplace=False, dry=True)

    # mask = (d2d_data.peak_area_PE > d2d_data.spe_position*1.5)
    # clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

    # sum_area_PE_list_0, rel_time_0 = get_sum_area_PE_in_time_window(
    counts = get_counts_in_time_window(
        clean_data.peak_start_time_s, 
        clean_data.peak_rel_start_time_s,
        clean_data.event_start_time_s, 
        clean_data.channel, 
        clean_data.board, 
        clean_data.peak_area_PE, 
        clean_data.event_id,
        time_window_width = 1,
        coincidence=2)
    count_list.append(counts)


In [ ]:
delay_time_list[np.argmax(count_list)]  # get the delay time with the maximum count

In [ ]:
plt.scatter(delay_time_list, count_list)
# plt.xlim(-5,1)

### Area spectrum

In [ ]:
delay_time = -1.9

change_time = d2d_data.get_df()
change_time.loc[change_time.board == 1, 'peak_start_time_s'] += delay_time
d2d_data_updated = d2d.data(change_time)

mask = (d2d_data_updated.peak_area_PE > d2d_data_updated.spe_position*1.5) 
# & (d2d_data_updated.peak_height_V < 1.2)
clean_data = d2d_data_updated.apply_mask(mask, inplace=False, dry=True)

# mask = (d2d_data.peak_area_PE > d2d_data.spe_position*1.5)
# clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

sum_area_PE_list_0, rel_time_0 = get_sum_area_PE_in_time_window(
    clean_data.peak_start_time_s, 
    clean_data.peak_rel_start_time_s,
    clean_data.event_start_time_s, 
    clean_data.channel, 
    clean_data.board, 
    clean_data.peak_area_PE, 
    clean_data.event_id,
    time_window_width = 0.05,
    coincidence=24)

In [ ]:
len(sum_area_PE_list_0 )

In [ ]:
plt.hist(sum_area_PE_list_0, bins=100
        #  , range=[-0.1,6000]
)
plt.xlabel('Sum Area [PE]')
plt.ylabel('Counts')

         
#log y
plt.yscale('log')

In [ ]:
mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==0)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_0, bin_0, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 0'
        #  , range = [49,51]
         )

mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==1)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_1, bin_1, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 1'
        #  , range = [49,51]
         )

print("Time Diff.", bin_1[:-1][cnt_1>1][0] - bin_0[:-1][cnt_0>1][0])

plt.legend()
         
#log y
plt.yscale('log')

In [ ]:
mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==0)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_0, bin_0, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 0'
         , range = [32,34]
         )

mask = (d2d_data.peak_area_PE > d2d_data.spe_position*300) & (d2d_data.board==1)
clean_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
cnt_1, bin_1, _ = plt.hist(clean_data.peak_start_time_s, bins=100, alpha=0.5, label='Board 1'
         , range = [32,34]
         )

print("Time Diff.", bin_1[:-1][cnt_1>1][0] - bin_0[:-1][cnt_0>1][0])

plt.legend()
         
#log y
plt.yscale('log')

In [ ]:
plt.hist(rel_time_0, bins=100
)
         
#log y
plt.yscale('log')

In [ ]:
np.max(sum_area_PE_list)

In [ ]:
# double check...

plt.close()
fig, ax = plt.subplots(figsize=(10,6))

# masking
for channel in range(24):

    mask = d2d_data.channel == channel
    singl_channel_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

    # this also remove null values from the peak_height_V_array
    array = list(singl_channel_data.peak_start_time_s_array)
    peak_start_time = np.concatenate(array)

    ax.hist(peak_start_time, range = [10.015,10.020], bins = 400, alpha=0.5, label=f"Channel {channel}", stacked=True,)

# Set plot title
# plt.suptitle(f"Dataset: {path.split('/')[-1]}")

# Move title upwards
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.legend()


plt.show()

### Time coincidence

In [ ]:
# double check...



plt.close()
fig, ax = plt.subplots(figsize=(10,6))

result_list = []

# masking
for channel in range(24):

    mask = d2d_data.channel == channel
    singl_channel_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

    # this also remove null values from the peak_height_V_array
    array = list(singl_channel_data.peak_start_time_s_array)
    peak_start_time = np.concatenate(array)

    np.hist(peak_start_time, range = [10.015,10.020], bins = 400, alpha=0.5, label=f"Channel {channel}", stacked=True,)

# Set plot title
# plt.suptitle(f"Dataset: {path.split('/')[-1]}")

# Move title upwards
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.legend()


plt.show()

### Test

In [ ]:
event_id = 3000
single_waveform = waveform[event_id,:]

baseline, baseline_std = get_baseline_for_all_events(waveform)
single_baseline, single_baseline_std = baseline[event_id], baseline_std[event_id]

event_time_s = waveform_processor.event_time_s[event_id]


In [ ]:
plt.scatter(points,single_waveform[points], color='r', label='boundaries of peaks', zorder=10)
plt.plot(single_waveform, label='filtered waveform')
plt.plot(smooth_waveform, label='waveform with rolling window')
plt.hlines(threshold, 0, 1000, 'black', linestyles='--', label='threshold')

plt.legend()
plt.xlabel('Sample')
plt.ylabel('Amplitude [V]')

In [ ]:
pairs = np.array([[15, 27], [38, 50], [60, 80]])
peak_width = pairs[:,1] - pairs[:,0]
tmp = np.where(peak_width < min_peak_width_sample)

pairs = np.delete(pairs, tmp, axis=0)
pairs

In [ ]:
min_peak_width_sample

In [ ]:
v_get_waveform = np.vectorize(
    get_peaks, 
    excluded=['window_size', 'threshold_sig', 'peak_width_sample'], 
    signature="(n) -> ()")

In [ ]:
single_waveform.shape

#### nd version

In [ ]:
axis = 1
window_size = 6
threshold_sig = 3
event_id = 3000

# single_waveform = waveform[event_id,:]
# single_baseline, single_baseline_std = baseline[event_id], baseline_std[event_id]

smooth_waveform = rolling_window(waveform, window_size, axis=1)
    
# add rolling window to smooth the data, roll every 3 points
# test_averaged = np.convolve(waveform, np.ones(window_size)/window_size, mode='valid')

# recalculated the baseline for the smoothed waveform
# baseline, baseline_std = get_baseline_for_all_events(smooth_waveform)
threshold = baseline + threshold_sig * baseline_std

threshold = np.repeat(threshold, smooth_waveform.shape[1]).reshape(smooth_waveform.shape[0], -1)
mask = smooth_waveform > threshold

# find where the waveform passes the threshold
diff = np.diff(mask, axis = 1)

points = np.where(diff == 1)
event_id, array_idx, count = np.unique(points[0], return_counts=True, return_index=True)

# remove the last point if it's odd
odd_points = np.where(count % 2 != 0)[0]

# since all repeating points are consecutive, so we can just add the count to the index
# This will give us the end index of the last point
last_index = array_idx + count - 1
last_index_to_remove = last_index[odd_points]

# remove the last point if it's odd
peaks_position_event = np.delete(points[0], last_index_to_remove)
peaks_position_sample = np.delete(points[1], last_index_to_remove)

# double check if the peaks_position_event is odd
event_id, array_idx, count = np.unique(peaks_position_event, return_counts=True, return_index=True)
assert len(np.where(count % 2 != 0)[0]) == 0, "There are still odd points in the peaks_position_event array."


In [ ]:
test_event_id = 5000
start_idx = array_idx[test_event_id]
end_idx = array_idx[test_event_id] + count[test_event_id]
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(threshold[test_event_id], 'b--', label='threshold')
ax.plot(mask[test_event_id], 'g--', label='threshold')
ax.plot(waveform[test_event_id,:], label='smoothed waveform')
ax.plot(smooth_waveform[test_event_id,:], label='smoothed waveform')

ax.plot(peaks_position_sample[start_idx:end_idx],
        test[start_idx:end_idx], 'ro', label='start of peak')

In [ ]:


# # mark the start and end of the peaks
# diff = np.diff(mask, axis = 1)
# points = np.where(diff == 1)[0]
# # start points are the points after the rising edge
# points[0::2] = points[0::2]+1 

# # # define the minimum peak width in samples
# # min_peak_width_sample = window_size*2

# # pair the start and end points of the peaks
# # remove the last point if it's odd
# if len(points) % 2 != 0:
#     points = points[:-1]  
# # reshape the points into pairs
# pairs = points.reshape(-1, 2)

# # remove the pair if they are too close to each other
# for pair in pairs:
#     if pair[1] - pair[0] < peak_width_sample:
#         pairs = np.delete(pairs, np.where((pairs == pair).all(axis=1)), axis=0)

# # flatten the pairs be better handling
# points = pairs.ravel()

# # results
# pairs_ns = pairs * 4 # in ns, assuming the sampling rate is 250 MHz (4 ns per sample)
# start_time_array, end_time_array = pairs_ns[:,0], pairs_ns[:,1]

# peak_max_array = np.empty(pairs.shape[0], dtype=float)
# area_array = np.empty(pairs.shape[0], dtype=float)

# for i, pair in enumerate(pairs):
#     peak_max_array[i] = np.max(waveform[pair[0]:pair[1]])
#     area_array[i] = np.sum(waveform[points[0]:points[1]])

# return start_time_array, end_time_array, peak_max_array, area_array

In [ ]:
event_id = np.random.randint(0, len(baseline), size=1)
event_id

#### 1d version

In [ ]:
event_id = np.random.randint(0, len(baseline), size=1)[0]

single_processed_waveform = waveform[event_id,:]
single_baseline = baseline[event_id]
single_baseline_std = baseline_std[event_id]
threshold_sig=5
extend_sum_window=50
        
window_size = 6
min_peak_width_sample = window_size*3

smooth_waveform = np.convolve(
    single_processed_waveform, 
    np.ones(window_size)/window_size, mode='valid')

# recalculated the baseline for the smoothed waveform
threshold = single_baseline + threshold_sig * single_baseline_std
print(threshold)


# applying the threshold to find peaks
mask = smooth_waveform > threshold

# mark the start and end of the peaks
diff = np.diff(mask)

samples_above_threshold = np.where(diff == 1)[0]
# start samples_above_threshold are the samples_above_threshold after the rising edge
# samples_above_threshold[0::2] = samples_above_threshold[0::2]+1 
samples_above_threshold[1::2] = samples_above_threshold[1::2] + window_size

# remove the last point if it's odd
if len(samples_above_threshold) % 2 != 0:
    samples_above_threshold = samples_above_threshold[:-1]  
    
# reshape the points into pairs for better handling
peak_boundaries = samples_above_threshold.reshape(-1, 2)

# remove the pair if they are too close to each other
peak_width = peak_boundaries[:,1] - peak_boundaries[:,0]
tmp = np.where(peak_width < min_peak_width_sample)
peak_boundaries = np.delete(peak_boundaries, tmp, axis=0)



fig, ax = plt.subplots(figsize=(10,6))
print(f"Threshold: {threshold:.3f} V")
# ax.plot(mask, 'g--', label='threshold')
ax.plot(single_processed_waveform, label='raw waveform')
ax.plot(smooth_waveform, label='smoothed waveform')
ax.axhline(threshold, color='black', linestyle = 'dashed', label='threshold')


for peak_id, peak_boundary in enumerate(peak_boundaries):
    while (extend_sum_window > 0):
        start_sample = np.max([peak_boundary[0]-extend_sum_window, 0])
        end_sample = np.min([peak_boundary[1]+extend_sum_window, len(single_processed_waveform)])

        # remove the pair if they are too far from the baseline
        y_diff = abs(single_processed_waveform[start_sample] - single_processed_waveform[end_sample])
        if y_diff > threshold_sig*single_baseline_std:
            extend_sum_window -= 10
        else:
            break
    else:
        # if we cannot find a valid window, just use the original peak boundary
        start_sample = peak_boundary[0]
        end_sample = peak_boundary[1]

    ax.plot([start_sample,end_sample],
        [single_processed_waveform[start_sample],single_processed_waveform[end_sample]], 
        'o', label = f"Peak {peak_id}")

plt.title(f"Event {event_id}")
plt.legend()
plt.xlabel('Sample')
plt.ylabel('Amplitude [V]')
plt.show()

In [ ]:
threshold

In [ ]:
y_diff = abs(single_processed_waveform[start_sample] - single_processed_waveform[end_sample])
y_diff

In [ ]:
np.min()

### Test

In [ ]:
axis = 1
window_size = 4
test = np.ones((99,200))
# rolled_array = np.lib.stride_tricks.sliding_window_view(test, 4, axis=1).mean(axis=1)
rolled_array = np.lib.stride_tricks.sliding_window_view(test, window_size, axis=axis)
test_mean = rolled_array.mean(axis=test.ndim) # this is the same as the rolling mean
# print(rolled_array)
print(test_mean.shape)
# rolled_array

In [ ]:
# # moved to script

# def get_peaks_for_single_waveform(single_waveform, baseline, baseline_std, threshold_sig=3):
#     axis = 1
#     window_size = 6
#     threshold_sig = 3
#     min_peak_width_sample = window_size*3

#     smooth_waveform = rolling_window(single_waveform, window_size, axis=0)
        
#     # recalculated the baseline for the smoothed waveform
#     threshold = baseline + threshold_sig * baseline_std

#     # applying the threshold to find peaks
#     mask = smooth_waveform > threshold

#     # mark the start and end of the peaks
#     diff = np.diff(mask)

#     samples_above_threshold = np.where(diff == 1)[0]
#     # start samples_above_threshold are the samples_above_threshold after the rising edge
#     # samples_above_threshold[0::2] = samples_above_threshold[0::2]+1 
#     samples_above_threshold[1::2] = samples_above_threshold[1::2] + window_size

#     # remove the last point if it's odd
#     if len(samples_above_threshold) % 2 != 0:
#         samples_above_threshold = samples_above_threshold[:-1]  
        
#     # reshape the points into pairs for better handling
#     peak_boundaries = samples_above_threshold.reshape(-1, 2)

#     # remove the pair if they are too close to each other
#     peak_width = peak_boundaries[:,1] - peak_boundaries[:,0]
#     tmp = np.where(peak_width < min_peak_width_sample)
#     peak_boundaries = np.delete(peak_boundaries, tmp, axis=0)

#     # results
#     peak_boundaries_ns = peak_boundaries * 4 # in ns, assuming the sampling rate is 250 MHz (4 ns per sample)
#     start_time_array_s, end_time_array_s = peak_boundaries_ns[:,0]/1e9, peak_boundaries_ns[:,1]/1e9

#     start_time_array_s += self.event_time_s
#     end_time_array_s += self.event_time_s

#     height_array_V = np.empty(peak_boundaries_ns.shape[0], dtype=float)
#     area_array_Vns = np.empty(peak_boundaries_ns.shape[0], dtype=float)
#     width_array_ns = np.empty(peak_boundaries_ns.shape[0], dtype=float)
#     area_array_Vns/self.spe_position


#     for i, peak_boundary in enumerate(peak_boundaries):
#         height_array_V[i] = np.max(single_waveform[peak_boundary[0]:peak_boundary[1]])
#         area_array_Vns[i] = np.sum(single_waveform[peak_boundary[0]:peak_boundary[1]])
#         width_array_ns[i] = peak_boundaries_ns[i,1] - peak_boundaries_ns[i,0] 

#     return start_time_array_s, end_time_array_s, height_array_V, area_array_Vns, width_array_ns
